In [24]:
# Importing usual libraries and setting color template

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) # to avoid deprecation warnings

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "vscode" # to be replaced by "iframe" if working on JULIE


In [25]:
dataset = pd.read_csv("https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/Decision+trees/uber.csv")


In [26]:
# Basic exploring and cleaning
# Display basic statistics about the dataset. 

print(f'Number of rows: {dataset.shape[0]}')
print()

print('Display of dataset: ')
display(dataset.head)
print()

print('Display of basic stats: ')
display(dataset.describe(include='all'))
print()

print('Percentage of missing values: ')
display(100*dataset.isnull().sum()/dataset.shape[0])

Number of rows: 20000

Display of dataset: 


<bound method NDFrame.head of        Unnamed: 0                            key  fare_amount  \
0        48462598    2015-05-07 10:24:44.0000004        13.00   
1         6637611    2014-07-09 09:14:04.0000002         5.50   
2         8357193  2013-11-11 18:51:00.000000240         8.50   
3        40466112   2014-05-22 01:54:00.00000069        19.00   
4        35405035    2011-06-21 23:37:33.0000002         7.70   
...           ...                            ...          ...   
19995    16885589   2010-07-29 23:59:00.00000084         8.90   
19996    25138434    2013-05-15 17:03:07.0000001        38.33   
19997    55176136  2012-03-08 18:39:00.000000211         5.70   
19998    16215179  2012-02-13 09:51:00.000000210         6.50   
19999    13391634    2010-12-19 21:06:01.0000003        14.90   

               pickup_datetime  pickup_longitude  pickup_latitude  \
0      2015-05-07 10:24:44 UTC        -73.971664        40.797035   
1      2014-07-09 09:14:04 UTC        -73.991635   


Display of basic stats: 


,Unnamed: 0,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,2.000000e+04,20000,20000.000000,20000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
unique,NaN,20000,NaN,19967,NaN,NaN,NaN,NaN,NaN
top,NaN,2015-05-07 10:24:44.0000004,NaN,2014-02-13 14:05:00 UTC,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,NaN,2,NaN,NaN,NaN,NaN,NaN
mean,2.767949e+07,NaN,11.358151,NaN,-72.490431,39.918498,-72.459891,39.923345,1.690150
std,1.601123e+07,NaN,9.891990,NaN,10.461597,6.051561,10.564266,6.901520,1.311384
min,3.949000e+03,NaN,-23.700000,NaN,-75.419276,-74.006190,-75.423067,-73.991765,0.000000
25%,1.383476e+07,NaN,6.000000,NaN,-73.992075,40.734733,-73.991423,40.734105,1.000000
50%,2.769724e+07,NaN,8.500000,NaN,-73.981904,40.752554,-73.980305,40.752997,1.000000
75%,4.148082e+07,NaN,12.500000,NaN,-73.967229,40.767075,-73.963509,40.768348,2.000000



Percentage of missing values: 


Unnamed: 0           0.0
key                  0.0
fare_amount          0.0
pickup_datetime      0.0
pickup_longitude     0.0
pickup_latitude      0.0
dropoff_longitude    0.0
dropoff_latitude     0.0
passenger_count      0.0
dtype: float64

In [27]:
# Some negative amount in the fare_amout column, shouldn't be possible.

In [28]:
# Drop the first two columns because useless for ML (id key, and unnamed column)
dataset = dataset.drop(["Unnamed: 0", "key"], axis = 1)

# Drop rows were fare_amount is negative
mask = dataset['fare_amount'] > 0
dataset = dataset.loc[mask, :]

print(f'Number of rows: {dataset.shape[0]}')
display(dataset.describe(include='all'))

Number of rows: 19998


,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,19998.000000,19998,19998.000000,19998.000000,19998.000000,19998.000000,19998.000000
unique,NaN,19965,NaN,NaN,NaN,NaN,NaN
top,NaN,2014-02-13 14:05:00 UTC,NaN,NaN,NaN,NaN,NaN
freq,NaN,2,NaN,NaN,NaN,NaN,NaN
mean,11.360472,NaN,-72.490283,39.918414,-72.459736,39.923259,1.690069
std,9.889051,NaN,10.462110,6.051858,10.564783,6.901859,1.311415
min,0.010000,NaN,-75.419276,-74.006190,-75.423067,-73.991765,0.000000
25%,6.000000,NaN,-73.992074,40.734729,-73.991422,40.734105,1.000000
50%,8.500000,NaN,-73.981904,40.752551,-73.980302,40.752997,1.000000
75%,12.500000,NaN,-73.967234,40.767075,-73.963506,40.768348,2.000000


In [29]:
# Convert the `pickup_datetime` column into datetime format.

dataset['pickup_datetime'] = pd.to_datetime(dataset['pickup_datetime'])

print(dataset["pickup_datetime"].dtype)


datetime64[us, UTC]


In [30]:
# create new columns: Year, Month, Day and Weekday
# Dropping 'pickup_datetime'ArithmeticError

dataset.loc[:, 'year'] = dataset['pickup_datetime'].dt.year
dataset.loc[:, 'month'] = dataset['pickup_datetime'].dt.month
dataset.loc[:, 'day'] = dataset['pickup_datetime'].dt.day

weekdays_dict = {
    0: 'Monday',
    1: 'Tuesday',
    2: 'Wednesday',
    3: 'Thursday',
    4: 'Friday',
    5: 'Saturday',
    6: 'Sunday'
}

dataset.loc[:, 'weekday'] = dataset['pickup_datetime'].dt.weekday.map(weekdays_dict)

dataset = dataset.drop('pickup_datetime', axis=1)

dataset.head()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day,weekday
0,13.0,-73.971664,40.797035,-73.958939,40.777649,1,2015,5,7,Thursday
1,5.5,-73.991635,40.749855,-73.988250,40.741341,2,2014,7,9,Wednesday
2,8.5,-73.982352,40.777042,-73.995912,40.759757,1,2013,11,11,Monday
3,19.0,-73.991455,40.751700,-73.936357,40.812327,1,2014,5,22,Thursday
4,7.7,-73.974749,40.756255,-73.952276,40.778332,1,2011,6,21,Tuesday


In [31]:
# Trying to know the ride distance using the GPS coordinates
# We need to use Haversine formula to do this
# Implementing a function to compute this formula for one ride using lon_1, lon_2, lat_1 and lat_2
# This function is not mine

def haversine(lon_1, lon_2, lat_1, lat_2):

    lon_1, lon_2, lat_1, lat_2 = map(np.radians, [lon_1, lon_2, lat_1, lat_2])  # Convert degrees to Radians


    diff_lon = lon_2 - lon_1
    diff_lat = lat_2 - lat_1


    distance_km = 2*6371*np.arcsin(np.sqrt(np.sin(diff_lat/2.0)**2 + np.cos(lat_1) * np.cos(lat_2) * np.sin(diff_lon/2.0)**2)) # earth radius: 6371km

    return distance_km


In [32]:
# Creating the column ride_distance

dataset['ride_distance'] = dataset.apply(lambda x: haversine(x['pickup_longitude'], x['dropoff_longitude'],
                                                             x['pickup_latitude'], x['dropoff_latitude']), axis=1)

dataset.head()


,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day,weekday,ride_distance
0,13.0,-73.971664,40.797035,-73.958939,40.777649,1,2015,5,7,Thursday,2.407225
1,5.5,-73.991635,40.749855,-73.988250,40.741341,2,2014,7,9,Wednesday,0.988729
2,8.5,-73.982352,40.777042,-73.995912,40.759757,1,2013,11,11,Monday,2.235651
3,19.0,-73.991455,40.751700,-73.936357,40.812327,1,2014,5,22,Thursday,8.183379
4,7.7,-73.974749,40.756255,-73.952276,40.778332,1,2011,6,21,Tuesday,3.099698


In [33]:
# Preprocessing
# Separating features X and target Y 

target_variable = "fare_amount"

X = dataset.drop(target_variable, axis=1)
Y = dataset.loc[:, target_variable]

print('Y :')
print(Y.head())
print()

print('X: ')
print(X.head())


Y :
0    13.0
1     5.5
2     8.5
3    19.0
4     7.7
Name: fare_amount, dtype: float64

X: 
   pickup_longitude  pickup_latitude  dropoff_longitude  dropoff_latitude  \
0        -73.971664        40.797035         -73.958939         40.777649   
1        -73.991635        40.749855         -73.988250         40.741341   
2        -73.982352        40.777042         -73.995912         40.759757   
3        -73.991455        40.751700         -73.936357         40.812327   
4        -73.974749        40.756255         -73.952276         40.778332   

   passenger_count  year  month  day    weekday  ride_distance  
0                1  2015      5    7   Thursday       2.407225  
1                2  2014      7    9  Wednesday       0.988729  
2                1  2013     11   11     Monday       2.235651  
3                1  2014      5   22   Thursday       8.183379  
4                1  2011      6   21    Tuesday       3.099698  


In [34]:
# Detect names of categorical and numerical features
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print('Numeric features: ', numeric_features)
print('Categorical features: ', categorical_features )

Numeric features:  ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'year', 'month', 'day', 'ride_distance']
Categorical features:  ['weekday']


In [35]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [36]:
# First, creating a baseline model with a multivariate linear regression
# Need to do the necessary transformations 

numeric_transformer = StandardScaler() # Only because of linear regression

categorical_transformer = OneHotEncoder(drop='first') # 0 missing values in the cat. features, so only need OHE

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [37]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(X_test) # Don't fit again !!
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore


Performing preprocessings on train set...
       pickup_longitude  pickup_latitude  dropoff_longitude  dropoff_latitude  \
8152         -73.966328        40.757692         -73.958325         40.768067   
1380         -73.988100        40.764807         -74.001052         40.746947   
14079        -74.003445        40.743692         -74.005457         40.738923   
7725         -73.961230        40.760852         -73.957473         40.722320   
14918        -73.966034        40.767998         -73.954902         40.783116   

       passenger_count  year  month  day    weekday  ride_distance  
8152                 1  2012      2   16   Thursday       1.336116  
1380                 1  2013     10   23  Wednesday       2.265861  
14079                1  2009      5   22     Friday       0.556722  
7725                 1  2013     11   10     Sunday       4.296238  
14918                1  2014      4    2  Wednesday       1.924727  
...Done.
[[-0.14344502  0.14080045 -0.14430518  0.1208558

In [38]:
# Training model on linear regression

regressor = LinearRegression()
regressor.fit(X_train, Y_train)

print('R2 score on train set: ', regressor.score(X_train, Y_train))
print('R2 score on test set: ', regressor.score(X_test, Y_test))

# After this, we can see the baseline model is very bad, pmretty much doesn't predict anything

R2 score on train set:  0.02419859579741468
R2 score on test set:  0.017058651115981927


In [39]:
# Now trying with a Random Forest model + default hyperparameters

regressor = RandomForestRegressor() # we must use a regressor here!
regressor.fit(X_train, Y_train)

print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))


# We see this model work far better, but we overfit (R2 on train set is way higher than R2 on test set)

R2 score on training set :  0.966687879127051
R2 score on test set :  0.7670457786612815


In [40]:
# Perform grid search
print("Grid search...")
regressor = RandomForestRegressor()

# Grid of values to be tested
params = {
    'max_depth': [10, 14],
    'min_samples_split': [8],
    'n_estimators': [80, 100]
}
gridsearch = GridSearchCV(regressor, param_grid = params, cv = 3, verbose = 2)
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)


Grid search...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END .max_depth=10, min_samples_split=8, n_estimators=80; total time=   1.7s
[CV] END .max_depth=10, min_samples_split=8, n_estimators=80; total time=   1.7s
[CV] END .max_depth=10, min_samples_split=8, n_estimators=80; total time=   1.7s
[CV] END max_depth=10, min_samples_split=8, n_estimators=100; total time=   2.2s
[CV] END max_depth=10, min_samples_split=8, n_estimators=100; total time=   2.1s
[CV] END max_depth=10, min_samples_split=8, n_estimators=100; total time=   2.2s
[CV] END .max_depth=14, min_samples_split=8, n_estimators=80; total time=   2.2s
[CV] END .max_depth=14, min_samples_split=8, n_estimators=80; total time=   2.2s
[CV] END .max_depth=14, min_samples_split=8, n_estimators=80; total time=   2.2s
[CV] END max_depth=14, min_samples_split=8, n_estimators=100; total time=   2.8s
[CV] END max_depth=14, min_samples_split=8, n_estimators=100; total time=   2.8s
[CV] END max_depth=14, min_samples

In [41]:
print('R2 score on training set: ', gridsearch.score (X_train, Y_train))
print('R2 score on test set: ', gridsearch.score(X_test, Y_test))

R2 score on training set:  0.9055525106915712
R2 score on test set:  0.7785497491160258


In [42]:
Y_train_pred = gridsearch.predict(X_train)
print(Y_train_pred)
print()

Y_test_pred = gridsearch.predict(X_test)
print(Y_test_pred)
print()

print('Mean Absolute Error on training set: ', round(mean_absolute_error(Y_train, Y_train_pred), 2))
print('Mean Fare on training set: ', round(Y_train.mean(), 2))
print()

print('Mean Absolute Error on test set: ', round(mean_absolute_error(Y_test, Y_test_pred), 2))
print('Mean Fare on test set: ', round(Y_test.mean(), 2))
print('Standard deviation on test set: ', round(Y_test.std(), 2)) 



[ 5.85805507  8.9047365   4.49318601 ...  7.30643905 25.05764042
 10.69053022]

[22.45786095  7.78896711 18.52794011 ...  5.06512084  8.54238817
  7.33587875]

Mean Absolute Error on training set:  1.52
Mean Fare on training set:  11.33

Mean Absolute Error on test set:  2.16
Mean Fare on test set:  11.48
Standard deviation on test set:  9.63


In [48]:
# So, the average fare is 11.48, withj a standard deviation of 9.63
# This model have a typical error of 2.16, which is good considering the mean and std

In [44]:
# bar plot to show the importance of each features 
column_names = []
for name, step, features_list in preprocessor.transformers_: # loop over steps of ColumnTransformer
    if name == 'num': # if pipeline is for numeric variables
        features = features_list # just get the names of columns to which it has been applied
    else: # if pipeline is for categorical variables
        features = step.get_feature_names_out() # get output columns names from OneHotEncoder
    column_names.extend(features) # concatenate features names

print("Names of columns corresponding to each coefficient: ", column_names)


Names of columns corresponding to each coefficient:  ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'year', 'month', 'day', 'ride_distance', 'weekday_Monday', 'weekday_Saturday', 'weekday_Sunday', 'weekday_Thursday', 'weekday_Tuesday', 'weekday_Wednesday']


In [45]:
# Create a pandas DataFrame
feature_importance = pd.DataFrame(index = column_names, data = gridsearch.best_estimator_.feature_importances_, columns=["feature_importances"])
feature_importance = feature_importance.sort_values(by = 'feature_importances')

# Plot coefficients
fig = px.bar(feature_importance, orientation = 'h')
fig.update_layout(showlegend = False,
                  margin = {'l': 120} # to avoid cropping of column names
                 )
fig.show()


In [46]:
# We see that to estimate the fare, the ride distance is by far the most important feature, wich seems logical 

In [ ]:
# i want to try the same model but without the ride distance
X = X.drop('ride_distance', axis = 1)



KeyError: "['ride_distance'] not found in axis"

In [50]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print('Numeric features: ', numeric_features)
print('Categorical features: ', categorical_features )

Numeric features:  ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'year', 'month', 'day']
Categorical features:  ['weekday']


In [51]:
# Divide dataset Train set & Test set
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5])
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(X_test) 
print('...Done.')
print(X_test[0:5,:])


Performing preprocessings on train set...
       pickup_longitude  pickup_latitude  dropoff_longitude  dropoff_latitude  \
8152         -73.966328        40.757692         -73.958325         40.768067   
1380         -73.988100        40.764807         -74.001052         40.746947   
14079        -74.003445        40.743692         -74.005457         40.738923   
7725         -73.961230        40.760852         -73.957473         40.722320   
14918        -73.966034        40.767998         -73.954902         40.783116   

       passenger_count  year  month  day    weekday  
8152                 1  2012      2   16   Thursday  
1380                 1  2013     10   23  Wednesday  
14079                1  2009      5   22     Friday  
7725                 1  2013     11   10     Sunday  
14918                1  2014      4    2  Wednesday  
...Done.
[[-0.14344502  0.14080045 -0.14430518  0.12085588 -0.52674365  0.14178757
  -1.22949054  0.03136581  0.          0.          0.          1

In [52]:
# Perform grid search
print("Grid search...")
regressor = RandomForestRegressor()

# Grid of values to be tested
params = {
    'max_depth': [10, 12, 14],
    'min_samples_split': [4, 8],
    'n_estimators': [60, 80, 100]
}
gridsearch = GridSearchCV(regressor, param_grid = params, cv = 3, verbose = 2)
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)


Grid search...
Fitting 3 folds for each of 18 candidates, totalling 54 fits
[CV] END .max_depth=10, min_samples_split=4, n_estimators=60; total time=   1.2s
[CV] END .max_depth=10, min_samples_split=4, n_estimators=60; total time=   1.2s
[CV] END .max_depth=10, min_samples_split=4, n_estimators=60; total time=   1.2s
[CV] END .max_depth=10, min_samples_split=4, n_estimators=80; total time=   1.6s
[CV] END .max_depth=10, min_samples_split=4, n_estimators=80; total time=   1.6s
[CV] END .max_depth=10, min_samples_split=4, n_estimators=80; total time=   1.6s
[CV] END max_depth=10, min_samples_split=4, n_estimators=100; total time=   2.0s
[CV] END max_depth=10, min_samples_split=4, n_estimators=100; total time=   2.0s
[CV] END max_depth=10, min_samples_split=4, n_estimators=100; total time=   2.0s
[CV] END .max_depth=10, min_samples_split=8, n_estimators=60; total time=   1.2s
[CV] END .max_depth=10, min_samples_split=8, n_estimators=60; total time=   1.1s
[CV] END .max_depth=10, min_sampl

In [54]:
print("R2 score on training set : ", gridsearch.score(X_train, Y_train))
print("R2 score on test set : ", gridsearch.score(X_test, Y_test))
print()

# Predictions on training set
print("Predictions on training set...")
Y_train_pred = gridsearch.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

# Predictions on test set
print("Predictions on test set...")
Y_test_pred = gridsearch.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

print("Mean Absolute Error on training set : ", mean_absolute_error(Y_train, Y_train_pred))
print("Mean Fare on training set : ", Y_train.mean())
print()
print("Mean Absolute Error on test set : ", mean_absolute_error(Y_test, Y_test_pred))
print("Mean Fare on test set : ", Y_test.mean())
print("Standard-deviation on test set : ", Y_test.std())


R2 score on training set :  0.8516908998911006
R2 score on test set :  0.7065365683293408

Predictions on training set...
...Done.
[ 7.53509691  8.62372891  7.47216958 ...  7.47472452 25.7736138
  8.70230136]

Predictions on test set...
...Done.
[13.77214491  8.6328336   7.39577424 ...  7.45968345  7.79979745
  7.95299052]

Mean Absolute Error on training set :  2.4251076380756933
Mean Fare on training set :  11.330150643830478

Mean Absolute Error on test set :  2.8663783546368657
Mean Fare on test set :  11.481740000000002
Standard-deviation on test set :  9.631479542206533


In [56]:
# bar plot to show the importance of each features 
column_names = []
for name, step, features_list in preprocessor.transformers_: # loop over steps of ColumnTransformer
    if name == 'num': # if pipeline is for numeric variables
        features = features_list # just get the names of columns to which it has been applied
    else: # if pipeline is for categorical variables
        features = step.get_feature_names_out() # get output columns names from OneHotEncoder
    column_names.extend(features) # concatenate features names

print("Names of columns corresponding to each coefficient: ", column_names)

# Create a pandas DataFrame
feature_importance = pd.DataFrame(index = column_names, data = gridsearch.best_estimator_.feature_importances_, columns=["feature_importances"])
feature_importance = feature_importance.sort_values(by = 'feature_importances')

# Plot coefficients
fig = px.bar(feature_importance, orientation = 'h')
fig.update_layout(showlegend = False,
                  margin = {'l': 120} # to avoid cropping of column names
                 )
fig.show()

Names of columns corresponding to each coefficient:  ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'year', 'month', 'day', 'weekday_Monday', 'weekday_Saturday', 'weekday_Sunday', 'weekday_Thursday', 'weekday_Tuesday', 'weekday_Wednesday']


In [57]:
# Even without the ride distance, the perform way better that the first baseline linear regressor.
# Probably because Random Forest handle non linear better ? 
# The model is able to use the GPS coordinates to get some informations related to the ride distance

# Still, predictions were better with the ride distance